In [1]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для поддержки Timestamp из паркета
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [2]:
# Создаем необходимы таблицы

## payments
spark.sql("""CREATE TABLE dds.payments(
  `id` BIGINT NOT NULL,
  `trip_id` BIGINT NOT NULL,
  `price` BIGINT,
  `tariff` BIGINT,
  `promo` BOOLEAN,
  `last_updated` DATE NOT NULL
)
USING iceberg""")

## trips
## Заменили типа для полей started_at и finished_at на DATE
spark.sql("""CREATE TABLE dds.trips (
  `id` BIGINT NOT NULL,
  `user_id` BIGINT NOT NULL,
  `scooter_hw_id` STRING,
  `started_at` DATE, 
  `finished_at` DATE,
  `start_lat` DOUBLE,
  `start_lon` DOUBLE,
  `finish_lat` DOUBLE,
  `finish_lon` DOUBLE,
  `distance` DOUBLE,
  `price` BIGINT,
  `last_updated` DATE NOT NULL
)
USING iceberg""")

## events
spark.sql("""CREATE TABLE dds.events(
  `user_id` BIGINT NOT NULL,
  `timestamp` BIGINT,
  `type_id` BIGINT NOT NULL,
  `last_updated` DATE NOT NULL
)
USING iceberg""")

DataFrame[]

In [16]:
spark.sql("TRUNCATE TABLE dds.trips")
spark.sql("""INSERT INTO dds.trips
  SELECT `id`,
  `user_id`,
  `scooter_hw_id`,
  to_date(from_unixtime(started_at/1000000000)) AS started_at,
  to_date(from_unixtime(finished_at/1000000000)) AS finished_at,
  `start_lat`,
  `start_lon`,
  `finish_lat`,
  `finish_lon`,
  `distance`,
  `price`,
  current_timestamp() as `last_updated`
    FROM stage.trips
    """)

DataFrame[]

In [7]:
# Загружаем данные

## payments
spark.sql("""INSERT INTO dds.payments
  SELECT `id`,
  `trip_id`,
  `price`,
  `tariff`,
  `promo`,
  current_timestamp() AS `last_updated`
  FROM stage.payments
    """)

## trips
## При выполнении добавляем трансформацию полей started_at и 
spark.sql("""INSERT INTO dds.trips
  SELECT `id`,
  `user_id`,
  `scooter_hw_id`,
  to_date(from_unixtime(started_at/1000000000)) AS started_at,
  to_date(from_unixtime(finished_at/1000000000)) AS finished_at,
  `start_lat`,
  `start_lon`,
  `finish_lat`,
  `finish_lon`,
  `distance`,
  `price`,
  current_timestamp() as `last_updated`
    FROM stage.trips
    """)

## Загружаем events
spark.sql("""INSERT INTO dds.events
  SELECT `user_id`,
  `timestamp`,
  `type_id`,
  current_timestamp() AS `last_updated`
    FROM stage.events
    """)

DataFrame[]

In [19]:
spark.sql("""SELECT min(started_at) as min_started_at,
    max(started_at) as max_started_at,
    min(finished_at) as min_finished_at,
    max(finished_at) as max_finished_at
  FROM dds.trips""").show()

+--------------+--------------+---------------+---------------+
|min_started_at|max_started_at|min_finished_at|max_finished_at|
+--------------+--------------+---------------+---------------+
|    2023-06-01|    2023-08-30|     2023-06-01|     2023-08-30|
+--------------+--------------+---------------+---------------+



In [18]:
spark.sql("""SELECT *
  FROM dds.trips""").show(10)

+---+-------+---------------+----------+-----------+---------------+---------------+---------------+---------------+------------------+-----+------------+
| id|user_id|  scooter_hw_id|started_at|finished_at|      start_lat|      start_lon|     finish_lat|     finish_lon|          distance|price|last_updated|
+---+-------+---------------+----------+-----------+---------------+---------------+---------------+---------------+------------------+-----+------------+
|  1|   1393|  GT-GXLV2-99D4|2023-06-01| 2023-06-01| 55.76074980015|37.613998000366|55.771709800018|37.618098300294|1644.9869999999992|    0|  2025-12-07|
|  2|   1987|    XM-1S-56D8X|2023-06-01| 2023-06-01|55.742680999971|37.651061899694|55.737762199979|37.601283200021| 4430.513000000002|    0|  2025-12-07|
|  3|    981|   SN-ES2-82XF4|2023-06-01| 2023-06-01|55.761562000052|37.642958999712|55.749707300062|37.589230200223| 4473.652999999999|    0|  2025-12-07|
|  4|   1629|SN-MAXG30-56D2G|2023-06-01| 2023-06-01| 55.74342999992|37